In [1]:
from torchvision import datasets, transforms
import numpy as np
import torch
from torch.utils.data.sampler import SubsetRandomSampler
import torch.nn as nn
import torch.nn.functional as F

Tamaño 64x64, 1k muestras, 25 epochs

# 1. Llevar el conjunto de datos al entorno

In [3]:
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

In [4]:
train_data = datasets.ImageFolder('datasets_procesados/size_64x64/sample_1k', transform=transform)

In [5]:
test_size = 0.2
valid_size = 0.2
batch_size = 16 # cuántas imágenes se procesan juntas a la vez

# For test
num_data = len(train_data)
indices_data = list(range(num_data)) # una lista con la posicion de cada imagen
np.random.shuffle(indices_data)
split_tt = int(np.floor(test_size * num_data)) # calcula cuntas imágenes va a destinar a test. Como test_size es 0.2, si hubieran 1000 imagenes, serían 200 imágenes.
train_idx, test_idx = indices_data[split_tt:], indices_data[:split_tt] # corta la lista en dos trozos: test y train.

# AQUI TENEMOS LOS ÍNDICES

# For Valid
num_train = len(train_idx)
# ERROR ??????  indices_train = list(range(num_train)) # Al hacer esto, se genera una lista con números desde 0 hasta el tamaño de train_idx 
# en lugar de indexar sobre los valores reales de train_idx
np.random.shuffle(train_idx)
split_tv = int(np.floor(valid_size * num_train))
train_final_idx = train_idx[split_tv:]
valid_idx = train_idx[:split_tv]
train_idx = train_final_idx  # Actualizas la lista final de train



# np.random.shuffle(indices_train)
#split_tv = int(np.floor(valid_size * num_train))
#train_idx, valid_idx = indices_train[split_tv:], indices_train[:split_tv]


# AQUI COGE LAS FOTOS CON LOS INDICES DE ANTES 
# define samplers for obtaining training and validation batches
train_sampler = SubsetRandomSampler(train_idx)
test_sampler = SubsetRandomSampler(test_idx)
valid_sampler = SubsetRandomSampler(valid_idx)


# Loaders contains the data in tuple format 
# (Image in form of tensor, label)

# NO LO ENTIENDO BIEN. DEVUELVE: lote = (imágenes, etiquetas). 
# - Imágenes: bloque numérico de 4 dimensiones que agrupa las fotos del lote: {Forma (Shape):  [batch_size, canales, alto, ancho]
# - Etiquetas:  lista con la clase real que corresponde a cada una de las 16 imágenes del paquete: [0,1,1,0,...]
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, sampler=train_sampler, num_workers=0)
valid_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, sampler=valid_sampler, num_workers=0)
test_loader = torch.utils.data.DataLoader(train_data, sampler=test_sampler, batch_size=batch_size, num_workers=0)

# variable representing classes of the images
classes = [0, 1]

# 2. Inicialización de la CNN

Inicializamos el modelo:

In [6]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # convolutional layer
        self.conv1 = nn.Conv2d(3, 16, 5) # capa convolucional. 3 canales de entrada (RGB), 16 filtros de extracción de características y un kernel de 5x5
        # max pooling layer
        self.pool = nn.MaxPool2d(2, 2) # capa de Max Pooling. Escanea regiones de 2x2 y se queda con el valor más alto, reduciendo el alto y el ancho de la imagen a la mitad.
        self.conv2 = nn.Conv2d(16, 32, 5)
        self.dropout = nn.Dropout(0.2) # apaga aleatoriamente el 20% de las neuronas durante el entrenamiento para evitar el sobreajuste
        # NOTA
        self.fc1 = nn.Linear(32 * 13 * 13, 256) # Primera capa densa. Conecta la salida de las convoluciones a 256 neuronas. El valor primer valor depende del tamaño de las imágenes.
        self.fc2 = nn.Linear(256, 84) # Segunda capa densa que reduce las 256 neuronas a 84.
        self.fc3 = nn.Linear(84, 2) # Capa final de salida. Emite 2 números (uno para la clase fake y otro para real)
        self.softmax = nn.LogSoftmax(dim=1) # Convierte la salida en probabilidades logarítmicas para clasificar el resultado
        
    def forward(self, x): # donde los datos entran por la red (x representa el lote de imágenes)
        # add sequence of convolutional and max pooling layers
        x = self.pool(F.relu(self.conv1(x))) # Aplica la primera convolución, pasa la activación ReLU y reduce el tamaño a la mitad con el pooling.
        x = self.pool(F.relu(self.conv2(x))) # Aplica la segunda convolución, activa con ReLU y vuelve a reducir a la mitad con pooling.   
        x = self.dropout(x) # Aplica la desactivación aleatoria del 20%.   
        x = x.view(-1, 32 * 13 * 13) # Flatten: Transforma la matriz 3D (mapa de características) en un vector 1D de una sola dimensión para poder pasárselo a las capas densas
        # Procesa los datos a través de las capas densas intermedias
        x = F.relu(self.fc1(x))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.softmax(self.fc3(x))
        return x
    
# create a complete CNN
model = Net()


# Loss function: encargada de medir qué tan mal o bien está prediciendo la red. Compara la salida del modelo con la respuesta real.
# criterion = torch.nn.CrossEntropyLoss()
criterion = torch.nn.NLLLoss()
# Optimizer: Descenso de Gradiente Estocástico.
optimizer = torch.optim.SGD(model.parameters(), lr = 0.003, momentum= 0.9) # lr = tasa de aprendizaje. momentum: no entiendo bien 

# 3. Entrenar el modelo

In [7]:
# Definimos la variable manualmente
train_on_gpu = torch.cuda.is_available()

# Si hay GPU, también enviamos el modelo a la tarjeta gráfica
if train_on_gpu:
    print('CUDA disponible. El entrenamiento se ejecutará en la GPU.')
    model.cuda()
else:
    print('CUDA no disponible. El entrenamiento se ejecutará en la CPU.')

CUDA no disponible. El entrenamiento se ejecutará en la CPU.


In [8]:
# number of epochs to train the model
n_epochs = 25 

valid_loss_min = np.inf # track change in validation loss

for epoch in range(1, n_epochs+1):

    # keep track of training and validation loss
    train_loss = 0.0
    valid_loss = 0.0
    
    # train the model #
    model.train()
    for data, target in train_loader:
        # move tensors to GPU if CUDA is available
        if train_on_gpu:
            data, target = data.cuda(), target.cuda()
        # clear the gradients of all optimized variables
        optimizer.zero_grad()
        # forward pass: compute predicted outputs by passing inputs to the model
        output = model(data)
        # calculate the batch loss
        loss = criterion(output, target)
        # backward pass: compute gradient of the loss with respect to model parameters
        loss.backward()
        # perform a single optimization step (parameter update)
        optimizer.step()
        # update training loss
        train_loss += loss.item()*data.size(0)
        
    # Validating the model
    model.eval()
    for data, target in valid_loader:
        if train_on_gpu:
            data, target = data.cuda(), target.cuda()
        # forward pass: compute predicted outputs by passing inputs to the model
        output = model(data)
        # calculate the batch loss
        loss = criterion(output, target)
        # update average validation loss 
        valid_loss += loss.item()*data.size(0)
    
    # calculate average losses
    #train_loss = train_loss/len(train_loader.dataset)
    #valid_loss = valid_loss/len(valid_loader.dataset)

    #CORREGIDO??
    train_loss = train_loss / len(train_sampler)
    valid_loss = valid_loss / len(valid_sampler)
        
    # print training/validation statistics 
    print('Epoch: {} \tTraining Loss: {:.6f} \tValidation Loss: {:.6f}'.format(
        epoch, train_loss, valid_loss))
    
    # save model if validation loss has decreased
    if valid_loss <= valid_loss_min:
        print('Validation loss decreased ({:.6f} --> {:.6f}).  Saving model ...'.format(
        valid_loss_min,
        valid_loss))
        torch.save(model.state_dict(), 'model_cifar.pt')
        valid_loss_min = valid_loss

Epoch: 1 	Training Loss: 0.695050 	Validation Loss: 0.693022
Validation loss decreased (inf --> 0.693022).  Saving model ...
Epoch: 2 	Training Loss: 0.691512 	Validation Loss: 0.688320
Validation loss decreased (0.693022 --> 0.688320).  Saving model ...
Epoch: 3 	Training Loss: 0.686588 	Validation Loss: 0.693931
Epoch: 4 	Training Loss: 0.681850 	Validation Loss: 0.668725
Validation loss decreased (0.688320 --> 0.668725).  Saving model ...
Epoch: 5 	Training Loss: 0.667546 	Validation Loss: 0.653245
Validation loss decreased (0.668725 --> 0.653245).  Saving model ...
Epoch: 6 	Training Loss: 0.666156 	Validation Loss: 0.655303
Epoch: 7 	Training Loss: 0.656231 	Validation Loss: 0.651363
Validation loss decreased (0.653245 --> 0.651363).  Saving model ...
Epoch: 8 	Training Loss: 0.635813 	Validation Loss: 0.645030
Validation loss decreased (0.651363 --> 0.645030).  Saving model ...
Epoch: 9 	Training Loss: 0.644702 	Validation Loss: 0.633830
Validation loss decreased (0.645030 --> 0.

# 4. Evaluación del modelo

In [9]:
# 1. Variables de control
test_loss = 0.0
class_correct = [0.0, 0.0]
class_total = [0.0, 0.0]
classes = ['fake', 'real']
batch_size = 16  # Cambiar este número si el batch_size es diferente

model.eval() # Modo evaluación

# Desactivamos el cálculo de gradientes para ahorrar memoria y acelerar la evaluación
with torch.no_grad():
    for data, target in test_loader:
        if train_on_gpu:
            data, target = data.cuda(), target.cuda()
            
        output = model(data)
        loss = criterion(output, target)
        test_loss += loss.item() * data.size(0)
        
        # Obtener la clase predicha
        _, pred = torch.max(output, 1)
        correct_tensor = pred.eq(target.data.view_as(pred))
        
        if train_on_gpu:
            correct = np.squeeze(correct_tensor.cpu().numpy())
        else:
            correct = np.squeeze(correct_tensor.numpy())
            
        # Acumular aciertos por clase
        for i in range(len(target)):
            label = target.data[i]
            class_correct[label] += correct[i].item() if correct.ndim > 0 else correct.item()
            class_total[label] += 1

# 2. Calcular pérdidas y precisión globales
# test_loss = test_loss / len(test_loader.dataset)
test_loss = test_loss / len(test_sampler)
print(f'Test Loss: {test_loss:.6f}\n')

# Precisiones por cada clase individual
for i in range(2):
    if class_total[i] > 0:
        acc = 100 * class_correct[i] / class_total[i]
        print(f'Test Accuracy of {classes[i]:>5s}: {acc:2.0f}% ({int(class_correct[i])}/{int(class_total[i])})')
    else:
        print(f'Test Accuracy of {classes[i]:>5s}: N/A (sin ejemplos)')

# Precisión global total
total_correct = np.sum(class_correct)
total_samples = np.sum(class_total)
overall_acc = 100. * total_correct / total_samples
print(f'\nTest Accuracy (Overall): {overall_acc:2.0f}% ({int(total_correct)}/{int(total_samples)})')

Test Loss: 0.574867

Test Accuracy of  fake: 65% (132/202)
Test Accuracy of  real: 79% (157/198)

Test Accuracy (Overall): 72% (289/400)
